#  LLM-as-a-Judge

## What this notebook does

We now go beyond training a classifier and ask a deeper question:

> Can an LLM (Groq) **reason** about which response is better,
> without ever seeing a single training example?





In [20]:

!pip install groq

###  Imports

In [21]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import re
import json

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)


from openai import OpenAI
# Label mapping — must match the mapping from the main notebook
#   0 => A wins
#   1 => B wins
#   2 => Tie
LABEL_NAMES = ['A wins', 'B wins', 'Tie']

print("Imports done.")

Imports done.


Loading the data and prepare a sample for judging

In [22]:
import os
import pandas as pd # Ensure pandas is imported as it might be used here before other cells

# Path to the training data file
train_file_path = 'train.csv'

train = pd.DataFrame() # Initialize an empty DataFrame as fallback
sample_df = pd.DataFrame() # Initialize sample_df here so it's always defined

# Check if the file exists
if not os.path.exists(train_file_path):
    print(f"Error: The file '{train_file_path}' was not found.")
    print("Cannot proceed without a valid training dataset.")
elif os.path.getsize(train_file_path) == 0:
    print(f"Error: The file '{train_file_path}' is empty.")
    print("Cannot proceed without a valid training dataset.")
else:
    try:
        # Load the training data (which has human labels we can compare against)
        train = pd.read_csv(train_file_path, engine='python')
        print(f"Successfully loaded {len(train)} rows from '{train_file_path}'.")

        # Recreating the label column
        def get_label(row):
            if row['winner_model_a'] == 1:
                return 0   # A wins
            elif row['winner_model_b'] == 1:
                return 1   # B wins
            else:
                return 2   # Tie

        train['label'] = train.apply(get_label, axis=1)

        # --- Sample selection ---
        # We used 100 examples — enough for meaningful statistics yet small enough
        # to keep API costs low (each call costs a small amount of tokens).
        # We stratified so we get a balanced mix of A wins / B wins / Tie.
        SAMPLE_SIZE  = 100     # total examples to judge
        PER_CLASS    = SAMPLE_SIZE // 3   # ~33 per class

        sample_parts = []
        for label_val in [0, 1, 2]:
            # Ensure there are enough samples in the subset
            subset_potential = train[train['label'] == label_val]
            if len(subset_potential) < PER_CLASS:
                print(f"Warning: Not enough samples for label {label_val}. Found {len(subset_potential)}, need {PER_CLASS}. Sampling all available.")
                subset = subset_potential
            else:
                #  using random_state=42 for reproducibility.
                subset = subset_potential.sample(
                    n=PER_CLASS, random_state=42
                )
            sample_parts.append(subset)

        if sample_parts:
            sample_df = pd.concat(sample_parts).reset_index(drop=True)
        else:
            print("Warning: No samples generated for sample_df.")

        print(f"Sample size: {len(sample_df)}")
        print("Class distribution in sample:")
        print(sample_df['label'].value_counts().sort_index())

        if not sample_df.empty:
            display(sample_df.head(3))
        else:
            print("Sample DataFrame is empty.")

    except pd.errors.ParserError as e:
        print(f"ParserError when reading '{train_file_path}': {e}")
        print("Please check if the CSV file is correctly formatted.")
        print("Cannot proceed without a valid training dataset.")
    except Exception as e:
        print(f"An unexpected error occurred while processing '{train_file_path}': {e}")
        print("Cannot proceed without a valid training dataset.")


Successfully loaded 57477 rows from 'train.csv'.
Sample size: 99
Class distribution in sample:
label
0    33
1    33
2    33
Name: count, dtype: int64


,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie,label
0,3689807178,claude-2.1,gpt-4-0613,"[""write text about HuggingFace""]","[""Here is a sample text about HuggingFace:\n\n...","[""HuggingFace is a pioneering technology compa...",1,0,0,0
1,411448197,gpt-3.5-turbo-0613,nous-hermes-2-mixtral-8x7b-dpo,"[""Rap like eminem""]","[""Yo, listen up, I'm about to drop some fire,\...","[""I'm sorry, but I am an AI and not capable of...",1,0,0,0
2,2056804621,claude-2.0,chatglm2-6b,"[""hey. how's it going?""]","[""I'm doing well, thanks for asking!""]","[""Hello! As an AI language model, I don't have...",1,0,0,0


### Defining The Judge Prompt

The prompt is the most important design decision in LLM-as-a-Judge.
A vague prompt produces vague, inconsistent verdicts.
A well-structured prompt produces reliable  verdicts.

Key design choices here:
- Tell the model its exact role
- Define what 'better' means (helpful, accurate, clear)
- Ask for reasoning BEFORE the verdict (chain-of-thought improves accuracy)
- Demand a structured final answer that is easy to parse

In [23]:
def build_judge_prompt(prompt_text, response_a, response_b):
    """
    Build the prompt we send to LLM for each example.

    Design principles:
    1. Clear role definition so that the LLM knows it is an impartial evaluator
    2. Explicit criteria  to specify what does 'better' mean?
    3. Chain-of-thought  reasoning before verdict improves accuracy
    4. Structured output VERDICT: A / VERDICT: B / VERDICT: Tie
       to make parsing reliable and unambiguous
    """

    max_chars = 4000
    prompt_text = str(prompt_text)[:max_chars]
    response_a  = str(response_a)[:max_chars]
    response_b  = str(response_b)[:max_chars]

    return f"""You are an impartial AI response evaluator.

USER QUESTION:
{prompt_text}

RESPONSE A:
{response_a}

RESPONSE B:
{response_b}

Evaluate which response is better based on accuracy, helpfulness,
clarity, and completeness.

Briefly explain your reasoning in 2-3 sentences.

YOU MUST END YOUR RESPONSE WITH EXACTLY ONE OF THESE THREE LINES:
VERDICT: A
VERDICT: B
VERDICT: Tie

Do not write anything after the VERDICT line.
Your response is invalid if it does not end with a VERDICT line."""

# Test the prompt on one example
test_prompt = build_judge_prompt(
    sample_df.iloc[0]['prompt'],
    sample_df.iloc[0]['response_a'],
    sample_df.iloc[0]['response_b']
)
print("Example judge prompt (first 800 chars):")
print(test_prompt[:800])
print("...")

Example judge prompt (first 800 chars):
You are an impartial AI response evaluator.

USER QUESTION:
["write text about HuggingFace"]

RESPONSE A:
["Here is a sample text about HuggingFace:\n\nHuggingFace is an AI startup focused on natural language processing (NLP). It was founded in 2016 by Clement Delangue and Julien Chaumond and is based in Brooklyn, New York. HuggingFace is best known for creating and open-sourcing Transformer-based machine learning models for NLP tasks like text classification, translation, summarization and question answering. \n\nSome key facts about HuggingFace:\n\n- HuggingFace gained popularity in the AI community for open-sourcing pretrained models like BERT, GPT-2, and T5. By making state-of-the-art models freely available, HuggingFace has helped democratize AI and enabled more research.\n\n- In addi
...


The Groq API call + response parser

In [ ]:
import os
import re
import time
from groq import Groq

#  Setting the API key as an environment variable
os.environ["GROQ_API_KEY"] = "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"

#  Initializing client (will automatically read GROQ_API_KEY)
client = Groq()

def parse_verdict(response_text):
    match = re.search(r'VERDICT:\s*(A|B|Tie)', response_text, re.IGNORECASE)
    if not match:
        return -1
    verdict = match.group(1).upper()
    return 0 if verdict == 'A' else 1 if verdict == 'B' else 2

def call_groq_judge(prompt_text, response_a, response_b, retries=3, seed=42):
    """
    Calls the Groq API to judge responses, with a seed for reproducibility.
    """
    judge_prompt = build_judge_prompt(prompt_text, response_a, response_b)

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                max_tokens=300,
                messages=[
                    {"role": "user", "content": judge_prompt}
                ],
                seed=42  # Add the seed parameter here for reproducibility
            )

            response_text = response.choices[0].message.content
            verdict = parse_verdict(response_text)
            return verdict, response_text

        except Exception as e:
            if "429" in str(e):  # rate limit
                wait_time = 2 ** attempt
                print(f"Rate limit hit, waiting {wait_time}s...")
                time.sleep(wait_time)
            else:
                print(f"API error on attempt {attempt+1}: {e}")
                time.sleep(1)

    return -1, "FAILED after retries"


# Testing if grok works with one example
print("Testing Groq judge on one example...")

test_verdict, test_reasoning = call_groq_judge(
    sample_df.iloc[0]['prompt'],
    sample_df.iloc[0]['response_a'],
    sample_df.iloc[0]['response_b']
)

print(f"\nGroq's reasoning:\n{test_reasoning}")
print(f"\nParsed verdict: {test_verdict} ({LABEL_NAMES[test_verdict] if test_verdict >= 0 else 'FAILED'})")
print(f"Human label:    {sample_df.iloc[0]['label']} ({LABEL_NAMES[sample_df.iloc[0]['label']]})")

### Running Groq on the full sample

This cell makes 100 API calls one per example.
We save results as we go so progress is not lost if it crashes midway.

In [25]:
# --- Running LLM-as-a-Judge on all 100 examples ---

groq_verdicts   = []   # numeric prediction from groq
groq_reasonings = []   # full text reasoning from groq
human_labels      = []   # ground truth from dataset

DELAY_BETWEEN_CALLS = 0.5  # seconds to avoid hitting rate limits (increased from 1s)

for i, row in sample_df.iterrows():
    # Progress indicator every 10 examples
    if len(groq_verdicts) % 10 == 0:
        print(f"  Processing {len(groq_verdicts)}/{len(sample_df)}...")

    verdict, reasoning = call_groq_judge(
        row['prompt'],
        row['response_a'],
        row['response_b']
    )

    groq_verdicts.append(verdict)
    groq_reasonings.append(reasoning)
    human_labels.append(row['label'])

    time.sleep(DELAY_BETWEEN_CALLS)

print(f"Done. Total judged: {len(groq_verdicts)}")

# Check how many failed to parse
failures = sum(1 for v in groq_verdicts if v == -1)
print(f"Parsing failures: {failures} ({failures/len(groq_verdicts)*100:.1f}%)")

# Save to CSV so we do not lose results
results_df = sample_df.copy()
results_df['groq_verdict']   = groq_verdicts
results_df['groq_reasoning'] = groq_reasonings
results_df.to_csv('llm_judge_results.csv', index=False)
print("Saved: llm_judge_results.csv")

  Processing 0/99...
Rate limit hit, waiting 1s...
Rate limit hit, waiting 2s...
Rate limit hit, waiting 4s...
Rate limit hit, waiting 1s...
Rate limit hit, waiting 2s...
Rate limit hit, waiting 4s...
Rate limit hit, waiting 1s...
Rate limit hit, waiting 2s...
Rate limit hit, waiting 4s...
Rate limit hit, waiting 1s...
Rate limit hit, waiting 2s...
Rate limit hit, waiting 4s...
Rate limit hit, waiting 1s...
Rate limit hit, waiting 2s...
Rate limit hit, waiting 4s...
Rate limit hit, waiting 1s...
Rate limit hit, waiting 2s...
Rate limit hit, waiting 4s...
Rate limit hit, waiting 1s...
Rate limit hit, waiting 2s...
Rate limit hit, waiting 4s...
Rate limit hit, waiting 1s...
Rate limit hit, waiting 2s...
Rate limit hit, waiting 4s...
Rate limit hit, waiting 1s...
Rate limit hit, waiting 2s...
Rate limit hit, waiting 4s...
Rate limit hit, waiting 1s...
Rate limit hit, waiting 2s...
Rate limit hit, waiting 4s...
  Processing 10/99...
Rate limit hit, waiting 1s...
Rate limit hit, waiting 2s.

KeyboardInterrupt: 

###  Evaluating Groq's agreement with human labels

In [ ]:
# --- Load results (in case this is run after the loop cell) ---
results_df = pd.read_csv('llm_judge_results.csv')

# Filter out failed parses for evaluation
valid_mask = results_df['groq_verdict'] != -1
valid_df   = results_df[valid_mask].copy()

print(f"Valid predictions: {valid_mask.sum()} / {len(results_df)}")

# --- Accuracy: how often does groq agree with humans? ---
groq_acc = accuracy_score(
    valid_df['label'],
    valid_df['groq_verdict']
)

print(f"\n=== LLM-as-a-Judge (groq) ===")
print(f"Agreement with human labels: {groq_acc:.4f} ({groq_acc*100:.1f}%)")
print(f"Random chance baseline:      {1/3:.4f} ({100/3:.1f}%)")
print()

# Confusion matrix
cm_groq = confusion_matrix(valid_df['label'], valid_df['groq_verdict'])
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm_groq, annot=True, fmt='d',
    xticklabels=LABEL_NAMES,
    yticklabels=LABEL_NAMES,
    cmap='Purples'
)
plt.title('Confusion Matrix — groq (LLM-as-a-Judge)')
plt.ylabel('Human Label (Ground Truth)')
plt.xlabel('groq Verdict')
plt.tight_layout()
plt.show()

### Inspecting disagreements  where do Groq and humans differ?

This seeks to understand
WHO is actually right when the Groq and the human disagree,
and gives insight about  human preference vs objective quality?

In [ ]:
# --- Find cases where groq disagrees with the human label ---
disagree_mask = valid_df['groq_verdict'] != valid_df['label']
disagree_df   = valid_df[disagree_mask].copy()

print(f"groq disagreed with humans on {disagree_mask.sum()} / {len(valid_df)} examples")
print(f"(That is the {100 - groq_acc*100:.1f}% it got 'wrong')")
print()

# Look at 3 disagreement examples with groq's full reasoning
# This is qualitative analysis — what was groq thinking?
print("=" * 70)
print("SAMPLE DISAGREEMENTS — reading these tells us WHY groq disagrees")
print("=" * 70)

for _, row in disagree_df.head(3).iterrows():
    human_lbl  = LABEL_NAMES[int(row['label'])]
    groq_lbl = LABEL_NAMES[int(row['groq_verdict'])]

    print(f"\nHuman label:    {human_lbl}")
    print(f"groq verdict: {groq_lbl}")
    print(f"\nPrompt (first 200 chars): {str(row['prompt'])[:200]}...")
    print(f"\ngroq's reasoning:")
    print(row['groq_reasoning'])
    print("-" * 70)

# KEY INSIGHT TO DISCUSS:
# Human preference is not always about 'objective quality'.
# Humans might prefer shorter answers, more confident tone,
# or responses that match their prior beliefs — even if technically less accurate.
# groq judges on quality; humans judge on preference. These can diverge.

In [ ]:
# --- Deeper inspection of disagreements ---
results_df = pd.read_csv('llm_judge_results.csv')

valid_mask = results_df['groq_verdict'] != -1
valid_df   = results_df[valid_mask].copy()

# Split into agreements and disagreements
agree_df    = valid_df[valid_df['groq_verdict'] == valid_df['label']]
disagree_df = valid_df[valid_df['groq_verdict'] != valid_df['label']]

print(f"Total valid: {len(valid_df)}")
print(f"Agreements:    {len(agree_df)}  ({len(agree_df)/len(valid_df)*100:.1f}%)")
print(f"Disagreements: {len(disagree_df)}  ({len(disagree_df)/len(valid_df)*100:.1f}%)")

# --- Where exactly does groq disagree? ---
# This tells us which human labels groq struggles with most
print("\nDisagreement breakdown by TRUE human label:")
for label_val, label_name in enumerate(LABEL_NAMES):
    subset = disagree_df[disagree_df['label'] == label_val]
    total  = valid_df[valid_df['label'] == label_val]
    if len(total) > 0:
        pct = len(subset) / len(total) * 100
        print(f"  When human said '{label_name}': "
              f"groq disagreed {len(subset)}/{len(total)} times ({pct:.1f}%)")



Stratified bias sampling

In [ ]:
results_df = pd.read_csv('llm_judge_results.csv')
valid_df   = results_df[results_df['groq_verdict'] != -1].copy()

# Split into LLaMA-involved and LLaMA-absent rows
llama_involved = valid_df[
    valid_df['model_a'].str.contains('llama', case=False, na=False) |
    valid_df['model_b'].str.contains('llama', case=False, na=False)
]
llama_absent = valid_df[
    ~valid_df['model_a'].str.contains('llama', case=False, na=False) &
    ~valid_df['model_b'].str.contains('llama', case=False, na=False)
]

acc_involved = accuracy_score(llama_involved['label'], llama_involved['groq_verdict'])
acc_absent   = accuracy_score(llama_absent['label'],   llama_absent['groq_verdict'])

print(f"Agreement when LLaMA IS a contestant:     {acc_involved:.1%}")
print(f"Agreement when LLaMA IS NOT a contestant: {acc_absent:.1%}")
print()

if acc_involved > acc_absent + 0.05:
    print("FINDING: Evidence of self-preference bias.")
    print("LLaMA agrees with humans more when LLaMA was the winner.")
elif acc_involved < acc_absent - 0.05:
    print("FINDING: Possible anti-LLaMA bias.")
    print("LLaMA is harder on its own family — possibly overcorrecting.")
else:
    print("FINDING: No strong evidence of self-preference bias.")
    print("Agreement rates are similar regardless of LLaMA involvement.")